# Build SDS archive and segment out event waveform files

## 1. Global parameters and imports


In [3]:
from pathlib import Path
from obspy import read, UTCDateTime, Stream
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.seisanio.utils.helpers import write_wavfile

TOP_PATH = Path('/Volumes') / 'tachyon' / 'Artemis2'
TOP_PATH = Path('/Volumes') / 'Pulwtop' / 'Artemis2'
SDS_DATA_PATH = TOP_PATH / '20_archive' / 'SDS'
WAV_PATH = TOP_PATH / '30_eventwav'

client = EnhancedSDSClient(str(SDS_DATA_PATH))  
launchtime = UTCDateTime('2026-04-01T22:35:12')
duration = 900
pretrigger = 1800
posttrigger = pretrigger
starttime=launchtime - pretrigger
endtime=launchtime + duration + posttrigger

def eventwav_path(launchtime, wav_path=WAV_PATH):
    return wav_path / f'{launchtime.year:04d}' / f'{launchtime.month:02d}'

## 2. USF Nanometrics stations
Already in an SDS archive, so just need to write out event file

In [ ]:
st = client.get_waveforms(network='1R', station='*', location='*', channel='D*', starttime=starttime, endtime=endtime)
write_wavfile(st, WAV_PATH, 'USF', len(st))

# 3. Guralp stations
Files downloaded through web interface

In [ ]:
GURALP_PATH = TOP_PATH / '00_download' / 'Guralp'

for station in ['B07', 'B12']:
    MSEED_PATH = GURALP_PATH / station
    station_files = sorted(list(MSEED_PATH.glob('*.mseed')))
    for station_file in station_files:
        print(f'Processing {station_file}...')
        st_station = read(str(station_file))
        client.write_stream(st_station)

Files downloaded with wget - it creates an sd/ folder

In [ ]:
MSEED_PATH = GURALP_PATH / 'B12' / 'sd'
station_files = sorted(list(MSEED_PATH.glob('*S1Seis*.mseed')))
for station_file in station_files:
    print(f'Processing {station_file}...')
    st_station = read(str(station_file))
    client.write_stream(st_station)

In [ ]:
st = client.get_waveforms(network='DG', station='*', location='*', channel='C*', starttime=starttime, endtime=endtime)
write_wavfile(st, WAV_PATH, 'MSFC', len(st))

# 4. BSU Gem sensors
Already copied all files from microSD cards to a single raw/ directory, and then ran gemconvert.
This created an mseed folder
Now we need to loop over the files in that folder, and write them to an SDS archive.

In [5]:
GEM_MSEED_PATH = TOP_PATH / '10_gemconvert' / 'mseed'
GEM_MSEED_PATH = TOP_PATH / '00_download' / 'gem' / 'mseed'
def fix_gem_trace_id(tr):
    id_dict = {
        '273': '1R.B24LT..HDF',
        '287': '1R.B24RE..HDF',
        '288': '1R.B24..HDF',
        '275': '1R.B24..HDI',
        '296': '1R.B29..HDF',
        '292': '1R.B29..HDI',
        '369': '1R.B12..HDF',
        '290': '1R.B23..HDF',
        '256': '1R.B03..HDF',
        '254': '1R.B03..HDI',
        '246': '1R.B14..HDF',
        '244': '1R.B14..HDI',
        '362': '1R.B01..HDF',
        '251': '1R.B01..HDI',
        '274': '1R.B07.10.HDF',
        '365': '1R.B07..HDF',
        '285': '1R.B20..HDF',
        '361': '1R.B20..HDI',
        '294': '1R.B20.10.HDI',
        '364': '1R.B20W..HDF',
        '366': '1R.B20S..HDF',
        '363': '1R.UNK..HD1' # what is 363, where was it?
    }
    tr.id = id_dict.get(tr.stats.station, tr.id)

for file in sorted(GEM_MSEED_PATH.glob('*.mseed')):
    st = read(str(file))
    for tr in st:
        fix_gem_trace_id(tr)
    print(st)
    print(f"Writing stream for file: {file}")
    client.write_stream(st)    

1 Trace(s) in Stream:
1R.B24..HDI | 2026-03-10T17:45:14.000000Z - 2026-03-10T23:59:59.990000Z | 100.0 Hz, 2248600 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-10T17_45_14..275..HDF.mseed
1 Trace(s) in Stream:
1R.B24..HDI | 2026-03-11T00:00:00.000000Z - 2026-03-11T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-11T00_00_00..275..HDF.mseed
1 Trace(s) in Stream:
1R.UNK..HD1 | 2026-03-11T14:50:43.000000Z - 2026-03-11T23:59:59.990000Z | 100.0 Hz, 3295700 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-11T14_50_43..363..HDF.mseed
1 Trace(s) in Stream:
1R.B24..HDI | 2026-03-12T00:00:00.000000Z - 2026-03-12T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-12T00_00_00..275..HDF.mseed


/opt/anaconda3/envs/flovopy_env/lib/python3.12/site-packages/obspy/io/mseed/core.py:1034: UserWarning: The encoding specified in trace.stats.mseed.encoding does not match the dtype of the data.
A suitable encoding will be chosen.
  warnings.warn(msg, UserWarning)


1 Trace(s) in Stream:
1R.UNK..HD1 | 2026-03-12T00:00:00.000000Z - 2026-03-12T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-12T00_00_00..363..HDF.mseed
1 Trace(s) in Stream:
1R.B24..HDI | 2026-03-13T00:00:00.000000Z - 2026-03-13T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-13T00_00_00..275..HDF.mseed
1 Trace(s) in Stream:
1R.UNK..HD1 | 2026-03-13T00:00:00.000000Z - 2026-03-13T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-13T00_00_00..363..HDF.mseed
1 Trace(s) in Stream:
1R.B24..HDI | 2026-03-14T00:00:00.000000Z - 2026-03-14T23:59:59.990000Z | 100.0 Hz, 8640000 samples
Writing stream for file: /Volumes/Pulwtop/Artemis2/00_download/gem/mseed/2026-03-14T00_00_00..275..HDF.mseed
1 Trace(s) in Stream:
1R.UNK..HD1 | 2026-03-14T00:00:00.000000Z - 2026-03-14

Now let's read the Gem data from the SDS archive - it is the only HD* channel data in there 

In [ ]:
st = client.get_waveforms(network='1R', station='*', location='*', channel='HD*', starttime=starttime, endtime=endtime)
write_wavfile(st, WAV_PATH, 'BSU', len(st))

# 5. Silicon Audio 1-minute files to SDS

In [ ]:
from pathlib import Path
from obspy import Stream, UTCDateTime, read
from flovopy.enhanced.sdsclient import EnhancedSDSClient

def merge_miniseed_to_sds(
    input_root,
    sds_root,
    pattern="*.ms",
    write_mode="merge",
    preprocess=False,
    verbose=True,
    merge_after_each_hour=False,
    **write_kwargs,
):
    """
    Read 1-minute MiniSEED files from a YYYY/MM/DD/HH directory structure,
    assemble them into day streams in strict chronological order, and write
    them into an SDS archive using EnhancedSDSClient.write_stream().

    Parameters
    ----------
    input_root : str or Path
        Root directory containing waveform files in YYYY/MM/DD/HH folders.
    sds_root : str or Path
        Root directory of the SDS archive to write into.
    pattern : str
        Glob pattern for waveform files inside each hour directory.
    write_mode : str
        SDS write mode passed to EnhancedSDSClient.write_stream():
        "fail", "overwrite", or "merge".
    preprocess : bool
        Passed through to EnhancedSDSClient.write_stream().
    verbose : bool
        If True, print progress information.
    merge_after_each_hour : bool
        If True, merge each hour stream before appending to the day stream.
        This can reduce memory pressure for very fragmented data.
    **write_kwargs
        Extra kwargs passed through to EnhancedSDSClient.write_stream().

    Returns
    -------
    list[Path]
        List of SDS files written.
    """
    input_root = Path(input_root)
    sds_root = Path(sds_root)

    sdsclient = EnhancedSDSClient(str(sds_root))
    written_all = []

    def log(msg):
        if verbose:
            print(msg)

    def sorted_numeric_dirs(parent):
        """
        Yield child directories whose names are digits, sorted numerically.
        """
        dirs = [p for p in parent.iterdir() if p.is_dir() and p.name.isdigit()]
        return sorted(dirs, key=lambda p: int(p.name))

    for year_dir in sorted_numeric_dirs(input_root):
        for month_dir in sorted_numeric_dirs(year_dir):
            for day_dir in sorted_numeric_dirs(month_dir):
                year = int(year_dir.name)
                month = int(month_dir.name)
                day = int(day_dir.name)

                day_start = UTCDateTime(year, month, day)
                day_end = day_start + 86400

                log(f"Processing {year:04d}-{month:02d}-{day:02d}")

                st_day = Stream()
                nfiles = 0

                for hour_dir in sorted_numeric_dirs(day_dir):
                    hour_files = sorted(hour_dir.glob(pattern), key=lambda p: p.name)

                    if not hour_files:
                        continue

                    if merge_after_each_hour:
                        st_hour = Stream()

                    for f in hour_files:
                        try:
                            st_file = read(str(f))
                            nfiles += 1

                            if merge_after_each_hour:
                                st_hour += st_file
                            else:
                                st_day += st_file

                        except Exception as e:
                            log(f"  Failed to read {f}: {e}")

                    if merge_after_each_hour and len(st_hour) > 0:
                        try:
                            st_hour.merge(method=0, fill_value=None)
                        except Exception as e:
                            log(f"  Hour-level merge failed in {hour_dir}: {e}")
                        st_day += st_hour

                if nfiles == 0:
                    log("  No files found")
                    continue

                if len(st_day) == 0:
                    log("  No valid traces")
                    continue

                # Merge only after the day has been assembled in chronological order.
                try:
                    st_day.merge(method=0, fill_value=None)
                except Exception as e:
                    log(f"  Day-level merge failed: {e}")
                    continue

                # Trim to exact UTC day bounds.
                st_day.trim(day_start, day_end, nearest_sample=False)

                if len(st_day) == 0:
                    log("  Nothing left after trimming to day bounds")
                    continue

                try:
                    written = sdsclient.write_stream(
                        st_day,
                        mode=write_mode,
                        preprocess=preprocess,
                        verbose=verbose,
                        **write_kwargs,
                    )
                    written_all.extend(written)
                    log(f"  Wrote {len(written)} SDS file(s)")
                except Exception as e:
                    log(f"  Failed to write SDS data for {year:04d}-{month:02d}-{day:02d}: {e}")

    return written_all


from pathlib import Path

B23_PATH = TOP_PATH / '10_downloads'  / 'SiliconAudio' / 'B23' / 'data'
SDS_B23 = Path('/Volumes') / 'tachyon' / '20260403_DOWNLOAD' / 'B23' / 'dayfiles'


written = merge_miniseed_to_sds(
    input_root=B23_PATH,
    sds_root=SDS_DATA_PATH,
    pattern="*.ms",
    write_mode="merge",
    preprocess=False,
    verbose=True,
)


We want to compute 1-s VSAM (and ISAM or PSAM) and then use the peak RMS, and convert it to SPL
Then plot SPL versus distance

We also want to plot spectrograms